# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a structured guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, describing survey-based mental health data collected in Kilifi County, Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata.to_json()

print(f"Dataset Title: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata.get('datePublished')}")

## 2. Data Overview
Review available record sets, fields and their `@id`s.

Croissant organizes tabular data into record sets, each with fields (columns). We'll list the available sets and fields with their unique `@id`s.

In [ ]:
# Retrieve record sets by @id

record_sets = dataset.metadata.get('recordSet', [])
if not record_sets:
    print('No record sets found in metadata.')
else:
    print('Available record sets and fields:')
    for rs in record_sets:
        rs_id = rs.get('@id') if isinstance(rs, dict) else rs
        print(f"- RecordSet @id: {rs_id}")
        rs_obj = dataset.metadata.find_by_id(rs_id)
        fields = rs_obj.get('field', []) if rs_obj else []
        if fields:
            for f in fields:
                f_id = f.get('@id') if isinstance(f, dict) else f
                print(f"    - Field @id: {f_id}")
        else:
            print("    (No fields listed)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
We'll use the record set and field `@id`s found above. For this notebook, we'll process the main survey record set (typically the first or principal set defined).

In [ ]:
# Extract data from available record sets
record_set_ids = []
for rs in dataset.metadata.get('recordSet', []):
    rs_id = rs.get('@id') if isinstance(rs, dict) else rs
    record_set_ids.append(rs_id)

dataframes = {}
for record_set_id in record_set_ids:
    # Use the generator for record extraction
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded records for RecordSet @id: {record_set_id}, shape: {dataframes[record_set_id].shape}")
        else:
            print(f"No records found for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Examine first RecordSet's fields
if dataframes:
    primary_record_set = list(dataframes.keys())[0]
    print(f"Columns in RecordSet @id {primary_record_set}: {dataframes[primary_record_set].columns.tolist()}")
    dataframes[primary_record_set].head()
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)
We'll process and analyze key numeric and categorical survey fields.

- Filter records based on mental health assessment scores (e.g., GAD-7 or PHQ-9).
- Normalize scores for comparison.
- Group records by demographic features (e.g., gender, education level).

All data elements are referenced strictly by their `@id`.

In [ ]:
# Identify numeric and grouping fields using their @id
primary_record_set_id = list(dataframes.keys())[0] if dataframes else None
df = dataframes.get(primary_record_set_id) if primary_record_set_id else None

# Example: search for a GAD-7 total score field by @id
# In this dataset, suppose total GAD-7 score field has @id 'https://api.app.sen.science/frontiers/7160411/gad7_total' (replace with the true one if different)
numeric_field_id = None
group_field_id = None

if df is not None:
    for col in df.columns:
        if 'gad' in col.lower() or 'phq' in col.lower():
            numeric_field_id = col
            break
    # Try gender or education for grouping
    for col in df.columns:
        if 'gender' in col.lower():
            group_field_id = col
            break
        elif 'education' in col.lower():
            group_field_id = col
            break
else:
    print('Primary DataFrame not found.')

print(f"Numeric field for analysis: {numeric_field_id}")
print(f"Grouping field: {group_field_id}")

# Filter records with high scores in chosen assessment
threshold = 10  # Example threshold
if numeric_field_id and numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by demographic feature if field exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean scores by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in DataFrame.")

## 5. Visualization
Visualize data distributions, e.g. GAD-7 scores by demographic group.

You may adjust field `@id`s as needed based on what's available.

In [ ]:
# Plot distribution of scores, grouped by demographic
if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
else:
    print('Fields for visualization not found or not loaded.')

## 6. Conclusion
- We've loaded a FAIR² Croissant-encoded mental health survey dataset, referencing all record sets and fields by their `@id`s.
- We filtered and normalized assessment scores (e.g., GAD-7), and grouped results by demographic fields to enable insights into mental health patterns in Kilifi County, Kenya.
- Visualizations highlighted how scores vary by group (e.g., gender, education).

Further analysis could enrich findings, including modeling, outlier exploration, and comparison to additional datasets.

All steps followed best practices for reproducible, AI-ready, FAIR² data processing.